# QFF2026 Phase 2 - PS-C1: H2 at R = 0.735 A
Run top to bottom from the repo root (`jupyter notebook main.ipynb`). Main study operator: Z2-tapered 2-qubit H2; JW-4q is reported as the baseline.

In [1]:
import os, sys
sys.path.insert(0, os.getcwd())
from c1.setup import *
from c1 import reduction
print('JW exact  :', exact_ground_energy(OP_JW, CONST), ' E_FCI:', E_FCI)
print('reduced   :', check_reduction(), ' (within 1e-6 Ha of E_FCI)')
print('reduction :', RED_INFO, 'HF label', HF_RED)

JW exact  : -1.1373060357534004  E_FCI: -1.1373060357534004
reduced   : -1.1373060357534004  (within 1e-6 Ha of E_FCI)
reduction : {'kept_qubits': [0, 2], 'sector': [-1, -1], 'pivots': [1, 3]} HF label 11


## 1. Mapping and reduction (c1_mapping.csv)
Symmetries: alpha-electron-number parity Z0Z1 and beta-electron-number parity Z2Z3 (HF sector eigenvalues -1, -1); qubits 1 and 3 are tapered.

In [2]:
from c1.resources import mapping_table
mapping_table(depth=1)

,operator_label,method,qubits,pauli_terms,qwc_groups,exact_energy,cx_on_D
0,JW-4q,none,4,14,5,-1.137306,3
1,tapered-2q,"Z2 tapering (alpha-parity, beta-parity)",2,4,2,-1.137306,1


## 2. Depth x optimizer x seed sweep on the tapered operator (c1_sweep.csv)

In [3]:
from c1.sweep import run_sweep, best_run, plot_best_convergence
df, hist, thetas = run_sweep(MAIN_LABEL, depths=(1, 2), tag='c1_sweep')
df

,operator,ansatz,depth,optimizer,seed,energy,error_mha,optimizer_calls,cx_on_D,depth_on_D
0,tapered-2q,hardware_efficient,1,COBYLA,0,-1.137306,5.806107e-07,68,1,9
1,tapered-2q,hardware_efficient,1,COBYLA,1,-1.137306,5.029128e-07,59,1,9
2,tapered-2q,hardware_efficient,1,COBYLA,2,-1.137306,1.432168e-06,64,1,9
3,tapered-2q,hardware_efficient,1,SPSA,0,-1.137306,3.060595e-05,651,1,9
4,tapered-2q,hardware_efficient,1,SPSA,1,-1.137306,1.707973e-04,651,1,9
5,tapered-2q,hardware_efficient,1,SPSA,2,-1.137306,8.518806e-05,651,1,9
6,tapered-2q,hardware_efficient,2,COBYLA,0,-1.137306,6.960970e-07,78,2,14
7,tapered-2q,hardware_efficient,2,COBYLA,1,-1.137306,9.705348e-08,61,2,14
8,tapered-2q,hardware_efficient,2,COBYLA,2,-1.137306,1.201775e-06,71,2,14
9,tapered-2q,hardware_efficient,2,SPSA,0,-1.137306,2.562342e-04,651,2,14


In [4]:
best = best_run(df, thetas)
print(best)
plot_best_convergence(df, hist, tag='c1_convergence', title='PS-C1 tapered 2q: best seed per depth x optimizer');

{'operator': 'tapered-2q', 'depth': 2, 'optimizer': 'COBYLA', 'seed': 1, 'error_mha': 9.705347636668193e-08, 'energy': -1.137306035656347, 'theta': [1.1216122077106316, 2.7194207092332987, -2.3149910176153194, 2.58358552539403, -1.2468895248556968, -0.4730586604571905]}


## 3. Shot-based estimation (c1_shots.csv)

In [5]:
from c1.shots import run_shot_study
run_shot_study(best)

,shots_per_group,condition,energy,std_error,error_mha,groups,total_shots,within_chem_acc_1se,within_chem_acc_point
0,1024,ideal,-1.141248,0.008230,-3.941866,2,2048,True,False
1,8192,ideal,-1.142315,0.002735,-5.009323,2,16384,False,False
2,1024,noisy_D,-1.111150,0.009481,26.155788,2,2048,False,False
3,8192,noisy_D,-1.107873,0.003302,29.432636,2,16384,False,False


## 4. Architecture: A, B, D (resources.csv)

In [6]:
from c1.resources import resources_table
resources_table(depths=(1, 2))

,ps,circuit_label,processor,executable,logical_qubits,physical_qubits,logical_cx,cx,depth,two_q_depth,swaps,cx_overhead
0,C1,JW-4q-HEA-depth1,A,True,4,5,3,3,16,3,0,0
1,C1,JW-4q-HEA-depth1,B,True,4,9,3,3,16,3,0,0
2,C1,JW-4q-HEA-depth1,D,True,4,4,3,3,16,3,0,0
3,C1,JW-4q-HEA-depth2,A,True,4,5,6,6,24,5,0,0
4,C1,JW-4q-HEA-depth2,B,True,4,9,6,6,24,5,0,0
5,C1,JW-4q-HEA-depth2,D,True,4,4,6,6,24,5,0,0
6,C1,tapered-2q-HEA-depth1,A,True,2,5,1,1,9,1,0,0
7,C1,tapered-2q-HEA-depth1,B,True,2,9,1,1,9,1,0,0
8,C1,tapered-2q-HEA-depth1,D,True,2,4,1,1,9,1,0,0
9,C1,tapered-2q-HEA-depth2,A,True,2,5,2,2,14,2,0,0


## 5. Objective 1: Bell test on Processor D (bell.csv)

In [7]:
from c1.bell import run_bell
run_bell()

,pair,pair_distance,condition,two_qubit_gates,XX,XX_se,YY,YY_se,ZZ,ZZ_se,fidelity,fidelity_se
0,0-1,1,ideal,1,1.000000,0.000000,-1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
1,0-1,1,noisy,1,0.945801,0.005074,-0.952148,0.004776,0.955078,0.004631,0.963257,0.002092
2,1-3,2,ideal,4,1.000000,0.000000,-1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
3,1-3,2,noisy,4,0.932129,0.005658,-0.935547,0.005519,0.955078,0.004631,0.955688,0.002290


## Additional experiment (labelled): JW-4q baseline sweep
Same settings on the unreduced 4-qubit operator; shows what the mapping choice costs in accuracy.

In [8]:
df4, hist4, th4 = run_sweep('JW-4q', depths=(1, 2), tag='c1_sweep_extra_JW4q')
plot_best_convergence(df4, hist4, tag='c1_convergence_extra_JW4q', title='Additional experiment: JW-4q');
df4

,operator,ansatz,depth,optimizer,seed,energy,error_mha,optimizer_calls,cx_on_D,depth_on_D
0,JW-4q,hardware_efficient,1,COBYLA,0,-1.116983,20.323031,200,3,16
1,JW-4q,hardware_efficient,1,COBYLA,1,-1.116970,20.336366,200,3,16
2,JW-4q,hardware_efficient,1,COBYLA,2,-1.116995,20.310911,200,3,16
3,JW-4q,hardware_efficient,1,SPSA,0,-1.116840,20.465823,651,3,16
4,JW-4q,hardware_efficient,1,SPSA,1,-1.116389,20.916680,651,3,16
5,JW-4q,hardware_efficient,1,SPSA,2,-1.116999,20.307398,651,3,16
6,JW-4q,hardware_efficient,2,COBYLA,0,-1.127244,10.062317,200,6,24
7,JW-4q,hardware_efficient,2,COBYLA,1,-1.118716,18.589698,200,6,24
8,JW-4q,hardware_efficient,2,COBYLA,2,-1.119428,17.878476,200,6,24
9,JW-4q,hardware_efficient,2,SPSA,0,-1.122130,15.176389,651,6,24
